<a href="https://colab.research.google.com/github/hyeseong-shin/algorithms/blob/main/202500639_%EC%8B%A0%ED%98%9C%EC%84%B1_11_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Practice Report7

EdgeWeightedGraph

Edge

In [13]:
from typing import List, Iterable, Tuple

class Edge:
  def __init__(self, v: int, w: int, weight: float):
    self._v = v
    self._w = w
    self._weight = weight

  def weight(self) -> float:
    return self._weight

  def either(self) -> int:
    return self._v

  def other(self, vertex: int) -> int: #반대편 vertex
    if vertex == self._v:
      return self._w
    elif vertex == self._w:
      return self._v
    else:
      raise ValueError("Inconsistent edge")

  def __repr__(self):
    return f"{self._v}-{self._w} {self._weight}"

In [14]:
class EdgeWeightedGraph:
  def __init__(self, V: int):
    self.V = V
    self.adj: list[list[Edge]] = [[] for _ in range(V)]
    """어떤 vertex의 인접 vertex들의 edge
    ex)Edge(0, 2, 1.5)를 추가
    adj[0] → [Edge(0, 2, 1.5)]
    adj[1] → []
    adj[2] → [Edge(0, 2, 1.5)]
    adj[3] → []"""

  def add_edge(self, e: Edge):
    v = e.either()
    w = e.other(v)
    self.adj[v].append(e)
    self.adj[w].append(e)

  def edges(self): #모든 edge 반환
    edge_list = []
    seen = set() #중복 제거

    for v in range(self.V):
      for e in self.adj[v]:
        if e not in seen:
          edge_list.append(e)
          seen.add(e)

    return edge_list

In [15]:
# build graph
G = EdgeWeightedGraph(4)
G.add_edge(Edge(0, 1, 1.0))
G.add_edge(Edge(0, 2, 4.0))
G.add_edge(Edge(1, 2, 2.0))
G.add_edge(Edge(1, 3, 5.0))
G.add_edge(Edge(2, 3, 1.0))
print(G.edges())

[0-1 1.0, 0-2 4.0, 1-2 2.0, 1-3 5.0, 2-3 1.0]


Kruskal's algorithm(MST)

UnionFind

In [16]:
class UnionFind:
  def __init__(self, n: int):
    self.parent = [i for i in range(n)] #초기상태: 모든 vertex가 자기자신을 parent로
    self.rank = [0] * n

  def find(self, x: int) -> int: #vertex가 속한 그룹의 root찾기
    if self.parent[x] != x: #ex) parent[2] = 1
      self.parent[x] = self.find(self.parent[x]) #ex) find(1)-> parent[2] = 0
     #=>parent[1] = 0 (path compression)

    return self.parent[x] #ex) parent[0]=0(root)->0반환

  def union(self, a: int, b: int): #a와 b가 속한 두 그룹을 하나로 합치기
    rootA = self.find(a)
    rootB = self.find(b)

    if rootA == rootB: #두 루트가 같으면 이미 같은 그룹으로 함수 종료
      return

    #attach the lower rank under the higher one
    if self.rank[rootA] < self.rank[rootB]:
      self.parent[rootA] = rootB

    elif self.rank[rootA] > self.rank[rootB]:
      self.parent[rootB] = rootA

    else: #같으면 A를 부모로 하고 rank +1
      self.parent[rootB] = rootA
      self.rank[rootA] += 1

  def connected(self, a: int, b: int) -> bool:#같은 그룹인지 확인
    return self.find(a) == self.find(b)

In [17]:
class KruskalMST:
  def __init__(self, G: EdgeWeightedGraph):
    self._mst: List[Edge] = [] #MST에 선택된 edge들
    self._weight: float = 0.0 #총 가중치 저장

    #1. sort all edges by weight
    edges = list(G.edges())
    edges.sort(key=lambda e: e.weight())

    uf = UnionFind(G.V)

    # find MST
    for e in edges:
      v = e.either() #edge의 한쪽 정점 반환
      w = e.other(v)

      if uf.connected(v, w):
        continue #cycle발생으로 피하기

      self._mst.append(e)
      self._weight += e._weight
      uf.union(v, w)

      if len(self._mst) == G.V - 1:
        break

  def edges(self) -> List[Edge]:
    return self._mst

  def weight(self) -> float:
    return self._weight

In [18]:
# build graph
G = EdgeWeightedGraph(4)
G.add_edge(Edge(0, 1, 1.0))
G.add_edge(Edge(0, 2, 4.0))
G.add_edge(Edge(1, 2, 2.0))
G.add_edge(Edge(1, 3, 5.0))
G.add_edge(Edge(2, 3, 1.0))
# Kruskal
kmst = KruskalMST(G)
print("Kruskal MST edges:", kmst.edges())
print("Kruskal total weight:", kmst.weight())

Kruskal MST edges: [0-1 1.0, 2-3 1.0, 1-2 2.0]
Kruskal total weight: 4.0
